# Train Route CVAE

This notebook launches training for the route conditional VAE using the existing training pipeline in the project.

## Colab setup (clone repo first)

If you run this notebook on a Google Colab kernel, run the next cell to clone the repo and set `PROJECT_ROOT`.

- Set `REPO_SLUG` to your GitHub repo (`owner/repo`).
- For private repos, provide a PAT with at least read access to repo contents.

In [21]:
import os
import ipywidgets as widgets
from IPython.display import display

os.environ.setdefault("REPO_SLUG", "ZYCC6002/Just-Go-Up-AI")

# create widget to load file into notebook kernel
widgets.FileUpload(
    accept='.env',  # .env file with 1 environment variable per line
    multiple=False  # True to accept multiple files upload else False
)

uploader = widgets.FileUpload()
display(uploader)

FileUpload(value={}, description='Upload')

In [22]:
filename_dict_key = list(uploader.value)[0]
env_content_string = uploader.value[filename_dict_key]['content'].decode('utf-8')

# splits each lines on the first '='
# and converts them into actual environment variables
for line in env_content_string.splitlines():
    if '=' in line and line.strip() and not line.strip().startswith('#'):
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        os.environ[key] = value

In [23]:
import os

# Set these in the Colab runtime before running the clone cell.
# Do not hardcode secrets into the notebook file.
REPO_SLUG = os.environ.get("REPO_SLUG", "").strip()
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()

if REPO_SLUG:
    print(f"REPO_SLUG set to: {REPO_SLUG}")
else:
    print("REPO_SLUG is not set yet.")

if GITHUB_TOKEN:
    print("GITHUB_TOKEN is already set in the runtime.")
else:
    print("GITHUB_TOKEN is not set yet.")

REPO_SLUG set to: ZYCC6002/Just-Go-Up-AI
GITHUB_TOKEN is already set in the runtime.


In [24]:
from pathlib import Path
import os
import sys
import subprocess

# Be strict to avoid false positives outside real Colab runtime
IN_COLAB = ("google.colab" in sys.modules) and Path("/content").exists()

REPO_SLUG = os.environ.get("REPO_SLUG", "").strip()
COLAB_REPO_DIR = Path("/content/JustGoUpAI")

def _build_git_auth_env_and_cleanup(token: str):
    """Return (env, cleanup_fn) for non-interactive authenticated git over HTTPS."""
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    askpass_path = None

    if token:
        askpass_path = Path("/tmp/git_askpass.sh")
        askpass_path.write_text(
            "#!/bin/sh\n"
            "case \"$1\" in\n"
            "  *Username*) echo \"x-access-token\" ;;\n"
            "  *Password*) echo \"$GITHUB_TOKEN\" ;;\n"
            "  *) echo \"\" ;;\n"
            "esac\n"
        )
        askpass_path.chmod(0o700)
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass_path)

    def _cleanup():
        if askpass_path is not None:
            askpass_path.unlink(missing_ok=True)
        env.pop("GITHUB_TOKEN", None)

    return env, _cleanup

def _run_git(cmd, repo_dir, env):
    return subprocess.run(
        cmd,
        cwd=str(repo_dir),
        check=True,
        timeout=90,
        text=True,
        capture_output=True,
        env=env,
    )

def run_git_pull(repo_dir, token: str):
    print(f"Updating repo in {repo_dir}...")
    env, cleanup = _build_git_auth_env_and_cleanup(token)
    try:
        try:
            # Fast-forward only avoids accidental merge commits in notebooks/ephemeral envs.
            _run_git(["git", "pull", "--ff-only"], repo_dir, env)
            print("Repo updated with fast-forward pull.")
            return True
        except subprocess.CalledProcessError as e_ff:
            print("Fast-forward pull failed.")
            if e_ff.stderr:
                print(e_ff.stderr.strip())

        try:
            # Fallback: rebase + autostash handles local drift safely in most cases.
            _run_git(["git", "pull", "--rebase", "--autostash"], repo_dir, env)
            print("Repo updated with rebase/autostash pull.")
            return True
        except subprocess.CalledProcessError as e_rb:
            print("Rebase pull also failed; keeping existing checkout and continuing.")
            if e_rb.stderr:
                print(e_rb.stderr.strip())
            print(
                "Tip: If this persists, open a terminal in Colab and run: "
                "git status && git branch --show-current && git remote -v"
            )
            return False
        except Exception as e:
            print(f"Unexpected git pull error: {e}")
            return False
    finally:
        cleanup()

if IN_COLAB:
    token = os.environ.get("GITHUB_TOKEN", "").strip()

    if COLAB_REPO_DIR.exists():
        print(f"Repo already exists at {COLAB_REPO_DIR}")
        if token:
            print("Using GitHub token from GITHUB_TOKEN env for git pull.")
        else:
            print("No GITHUB_TOKEN set. Pull will only work for public repos.")
        run_git_pull(COLAB_REPO_DIR, token)
    else:
        if not REPO_SLUG:
            raise RuntimeError("Set REPO_SLUG before running the clone cell.")

        clone_url = f"https://github.com/{REPO_SLUG}.git"
        env, cleanup = _build_git_auth_env_and_cleanup(token)

        if token:
            print("Using GitHub token from GITHUB_TOKEN env.")
        else:
            print("No GITHUB_TOKEN set. Attempting anonymous clone (public repos only).")
            print("If this is a private repo, set GITHUB_TOKEN in the Colab runtime and rerun this cell.")

        try:
            subprocess.run(
                ["git", "clone", clone_url, str(COLAB_REPO_DIR)],
                check=True,
                env=env,
                timeout=180,
            )
        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                "git clone failed. Check REPO_SLUG and, for private repos, set GITHUB_TOKEN in the runtime first."
            ) from e
        except subprocess.TimeoutExpired as e:
            raise RuntimeError(
                "git command timed out. Check network connectivity in Colab and rerun."
            ) from e
        finally:
            cleanup()

        print(f"Cloned repo to {COLAB_REPO_DIR}")

    os.environ["PROJECT_ROOT"] = str(COLAB_REPO_DIR)
    print(f"PROJECT_ROOT set to: {os.environ['PROJECT_ROOT']}")
else:
    print("Not running on Colab; skipping clone step.")

Using GitHub token from GITHUB_TOKEN env.
Cloned repo to /content/JustGoUpAI
PROJECT_ROOT set to: /content/JustGoUpAI


In [25]:
from pathlib import Path
import os
import sys
import shlex
import subprocess

IN_COLAB = "google.colab" in sys.modules

def is_project_root(p: Path) -> bool:
    return (p / "src/model_training/train_route_cvae.py").exists() and (p / "pyproject.toml").exists()

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]

    # Local workspace fallback
    candidates.append(Path("/Users/christopherchan/Documents/Coding Projects/2026/JustGoUpAI"))

    # Optional env override
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        env_path = Path(env_root).expanduser().resolve()
        candidates.extend([env_path, *env_path.parents])

    # PWD env fallback
    pwd_env = os.environ.get("PWD")
    if pwd_env:
        pwd_path = Path(pwd_env).resolve()
        candidates.extend([pwd_path, *pwd_path.parents])

    # Colab common locations
    if IN_COLAB:
        candidates.extend(
            [
                Path("/content/JustGoUpAI"),
                Path("/content/drive/MyDrive/JustGoUpAI"),
            ]
        )

    seen = set()
    uniq_candidates = []
    for c in candidates:
        c = c.resolve()
        if c not in seen:
            seen.add(c)
            uniq_candidates.append(c)

    for p in uniq_candidates:
        if is_project_root(p):
            return p

    raise FileNotFoundError(
        "Could not locate project root. "
        f"CWD={start}. "
        "If running on Colab, clone/copy the repo under /content and set os.environ['PROJECT_ROOT']."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Running on Colab: {IN_COLAB}")
print(f"CWD: {Path.cwd().resolve()}")
print(f"PWD env: {os.environ.get('PWD', '<none>')}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Src root: {SRC_ROOT}")
print(f"Training script exists: {(PROJECT_ROOT / 'src/model_training/train_route_cvae.py').exists()}")

Running on Colab: True
CWD: /content
PWD env: /
Project root: /content/JustGoUpAI
Src root: /content/JustGoUpAI/src
Training script exists: True


In [26]:
from pathlib import Path
import os

# In Colab, mount Google Drive and point DATA_ROOT at your Kilter_data folder.
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/content/drive/MyDrive/Kilter_data"))
    except Exception as e:
        raise RuntimeError(
            "Google Drive mount failed in Colab. "
            "Make sure you are running in a Colab runtime and have authorized Drive access. "
            "If your data is elsewhere in Drive, set DATA_ROOT to that folder path before running this cell."
        ) from e
else:
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT / "data"))

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"DB exists: {(DATA_ROOT / "raw/kilter_database.sqlite").exists()}")

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/Kilter_data
DB exists: True


In [27]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"MPS available (Apple): {torch.backends.mps.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 1
GPU name: Tesla T4
MPS available (Apple): False


## Training configuration

Adjust values below, then run the next cells.

In [28]:
train_cfg = {
    "db_path": DATA_ROOT / "raw/kilter_database.sqlite",
    "cache_path": DATA_ROOT / "preprocessed_routes_cache.pt",
    "checkpoint_path": DATA_ROOT / "route_cvae.pt",
    "loss_plot_path": DATA_ROOT / "route_cvae_loss_curve.png",

    # Data sampling / filtering
    "max_routes": 1000000,
    "min_holds": 1,
    "require_full_metadata": False,

    # Optimization
    "epochs": 20,
    "batch_size": 16,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "latent_dim": 32,
    "numeric_weight": 0.25,
    "kl_beta": 1,
    "kl_warmup_epochs": None,  # None -> auto 40% of epochs
    "grad_clip_norm": 1.0,     # Max magnitude of gradient updates
    "seed": 42,

    # Conditioning experiment knobs (quick disentanglement test)
    "encoder_use_condition": False,
    "encoder_use_cond_adaln": True,
    "decoder_use_cond_adaln": True,
    "decoder_z_memory_tokens": 8,

    # Early stopping
    "early_stop_delta": 0.01,
    "early_stop_patience": 2,

    # Resume and cache controls
    "resume": False,
    "resume_path": None,       # Defaults to checkpoint_path
    "rebuild_cache": False,
}

train_cfg

{'db_path': PosixPath('/content/drive/MyDrive/Kilter_data/raw/kilter_database.sqlite'),
 'cache_path': PosixPath('/content/drive/MyDrive/Kilter_data/preprocessed_routes_cache.pt'),
 'checkpoint_path': PosixPath('/content/drive/MyDrive/Kilter_data/route_cvae.pt'),
 'loss_plot_path': PosixPath('/content/drive/MyDrive/Kilter_data/route_cvae_loss_curve.png'),
 'max_routes': 1000000,
 'min_holds': 1,
 'require_full_metadata': False,
 'epochs': 20,
 'batch_size': 16,
 'lr': 0.0002,
 'weight_decay': 0.0001,
 'latent_dim': 32,
 'numeric_weight': 0.25,
 'kl_beta': 1,
 'kl_warmup_epochs': None,
 'grad_clip_norm': 1.0,
 'seed': 42,
 'encoder_use_condition': False,
 'encoder_use_cond_adaln': True,
 'decoder_use_cond_adaln': True,
 'decoder_z_memory_tokens': 8,
 'early_stop_delta': 0.01,
 'early_stop_patience': 2,
 'resume': False,
 'resume_path': None,
 'rebuild_cache': False}

In [29]:
import shlex

def build_train_command(cfg: dict) -> list[str]:
    """Build the subprocess command list for train_route_cvae.py from a config dict."""
    script = PROJECT_ROOT / "src/model_training/train_route_cvae.py"
    cmd = [sys.executable, "-u", str(script)]
    cmd += ["--db-path", str(cfg["db_path"])]
    cmd += ["--cache-path", str(cfg["cache_path"])]
    cmd += ["--checkpoint-path", str(cfg["checkpoint_path"])]
    cmd += ["--loss-plot-path", str(cfg["loss_plot_path"])]
    cmd += ["--max-routes", str(cfg["max_routes"])]
    cmd += ["--min-holds", str(cfg["min_holds"])]
    cmd += ["--epochs", str(cfg["epochs"])]
    cmd += ["--batch-size", str(cfg["batch_size"])]
    cmd += ["--lr", str(cfg["lr"])]
    cmd += ["--weight-decay", str(cfg["weight_decay"])]
    cmd += ["--latent-dim", str(cfg["latent_dim"])]
    cmd += ["--numeric-weight", str(cfg["numeric_weight"])]
    cmd += ["--kl-beta", str(cfg["kl_beta"])]
    cmd += ["--grad-clip-norm", str(cfg["grad_clip_norm"])]
    cmd += ["--seed", str(cfg["seed"])]

    if cfg.get("require_full_metadata"):
        cmd += ["--require-full-metadata"]
    if cfg.get("resume"):
        cmd += ["--resume"]
    if cfg.get("resume_path"):
        cmd += ["--resume-path", str(cfg["resume_path"])]
    if cfg.get("rebuild_cache"):
        cmd += ["--rebuild-cache"]
    if cfg.get("kl_warmup_epochs") is not None:
        cmd += ["--kl-warmup-epochs", str(cfg["kl_warmup_epochs"])]
    if cfg.get("early_stop_delta") is not None:
        cmd += ["--early-stop-delta", str(cfg["early_stop_delta"])]
    if cfg.get("early_stop_patience") is not None:
        cmd += ["--early-stop-patience", str(cfg["early_stop_patience"])]

    if cfg.get("encoder_use_condition", True):
        cmd += ["--encoder-use-condition"]
    else:
        cmd += ["--no-encoder-use-condition"]

    if cfg.get("encoder_use_cond_adaln", True):
        cmd += ["--encoder-use-cond-adaln"]
    else:
        cmd += ["--no-encoder-use-cond-adaln"]

    if cfg.get("decoder_use_cond_adaln", True):
        cmd += ["--decoder-use-cond-adaln"]
    else:
        cmd += ["--no-decoder-use-cond-adaln"]

    cmd += ["--decoder-z-memory-tokens", str(cfg.get("decoder_z_memory_tokens", 4))]
    return cmd


# Build main training command and display it
cmd = build_train_command(train_cfg)
print("Training command:")
print(" \n".join(shlex.quote(x) for x in cmd))

Training command:
/usr/bin/python3 
-u 
/content/JustGoUpAI/src/model_training/train_route_cvae.py 
--db-path 
/content/drive/MyDrive/Kilter_data/raw/kilter_database.sqlite 
--cache-path 
/content/drive/MyDrive/Kilter_data/preprocessed_routes_cache.pt 
--checkpoint-path 
/content/drive/MyDrive/Kilter_data/route_cvae.pt 
--loss-plot-path 
/content/drive/MyDrive/Kilter_data/route_cvae_loss_curve.png 
--max-routes 
1000000 
--min-holds 
1 
--epochs 
20 
--batch-size 
16 
--lr 
0.0002 
--weight-decay 
0.0001 
--latent-dim 
32 
--numeric-weight 
0.25 
--kl-beta 
1 
--grad-clip-norm 
1.0 
--seed 
42 
--early-stop-delta 
0.01 
--early-stop-patience 
2 
--no-encoder-use-condition 
--encoder-use-cond-adaln 
--decoder-use-cond-adaln 
--decoder-z-memory-tokens 
8


In [30]:
import os
import time
from pathlib import Path

benchmark_root = Path(PROJECT_ROOT) / "data" / "cache_speed_benchmark"
benchmark_root.mkdir(parents=True, exist_ok=True)

BENCHMARK_EPOCHS = 1
BENCHMARK_MAX_ROUTES = min(int(train_cfg["max_routes"]), 2000)

benchmark_cfg = {
    **train_cfg,
    "cache_path": benchmark_root / "preprocessed_routes_cache.pt",
    "checkpoint_path": benchmark_root / "route_cvae_benchmark.pt",
    "loss_plot_path": benchmark_root / "route_cvae_benchmark_loss_curve.png",
    "epochs": BENCHMARK_EPOCHS,
    "max_routes": BENCHMARK_MAX_ROUTES,
}


def run_timed_training(*, rebuild_cache: bool) -> float:
    cfg = {**benchmark_cfg, "rebuild_cache": rebuild_cache}
    cmd = build_train_command(cfg)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    start = time.perf_counter()
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, text=True, capture_output=True)
    elapsed = time.perf_counter() - start
    if proc.returncode != 0:
        raise RuntimeError(
            f"Benchmark failed (rebuild_cache={rebuild_cache}) exit={proc.returncode}\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return elapsed


print(f"Benchmarking with {BENCHMARK_MAX_ROUTES} routes and {BENCHMARK_EPOCHS} epoch(s)...")
uncached_s = run_timed_training(rebuild_cache=True)
cached_s = run_timed_training(rebuild_cache=False)
speedup_pct = (uncached_s - cached_s) / uncached_s * 100.0

print(f"Without cache: {uncached_s:.2f}s")
print(f"With cache:    {cached_s:.2f}s")
print(f"Speedup:       {speedup_pct:.1f}%")

Benchmarking with 2000 routes and 1 epoch(s)...
Without cache: 31.91s
With cache:    11.92s
Speedup:       62.6%


In [ ]:
# Optional: run a quick 3-run disentanglement sweep and compare validation/test loss.
# Requires build_train_command to be defined (run the cell above first).

import copy
import os
import re
import subprocess
from pathlib import Path

sweep_runs = [
    {
        "name": "baseline",
        "encoder_use_condition": True,
        "encoder_use_cond_adaln": True,
        "decoder_use_cond_adaln": True,
        "decoder_z_memory_tokens": 4,
    },
    {
        "name": "disentangled",
        "encoder_use_condition": False,
        "encoder_use_cond_adaln": True,
        "decoder_use_cond_adaln": True,
        "decoder_z_memory_tokens": 4,
    },
    {
        "name": "disentangled_z8",
        "encoder_use_condition": False,
        "encoder_use_cond_adaln": True,
        "decoder_use_cond_adaln": True,
        "decoder_z_memory_tokens": 8,
    },
]


def parse_losses_from_output(lines: list[str]) -> dict:
    val_total = best_val = test_total = None
    for line in lines:
        if m := re.search(r"val_total=([0-9]+\.[0-9]+)", line):
            val_total = float(m.group(1))
        if m := re.search(r"test_total=([0-9]+\.[0-9]+)", line):
            test_total = float(m.group(1))
        if m := re.search(r"best_val=([0-9]+\.[0-9]+)", line):
            best_val = float(m.group(1))
    return {"best_val": best_val, "final_val_total": val_total, "test_total": test_total}


summary = []
run_artifact_root = Path(train_cfg["checkpoint_path"]).parent / "sweep_runs"
run_artifact_root.mkdir(parents=True, exist_ok=True)

for run in sweep_runs:
    run_cfg = copy.deepcopy(train_cfg)
    run_cfg.update(run)
    run_name = run["name"]
    run_cfg["checkpoint_path"] = run_artifact_root / f"route_cvae_{run_name}.pt"
    run_cfg["loss_plot_path"] = run_artifact_root / f"route_cvae_{run_name}_loss.png"
    run_cfg["resume"] = False
    run_cfg["resume_path"] = None

    cmd_run = build_train_command(run_cfg)
    print(f"\n=== Running: {run_name} ===")

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd_run,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    log_lines = []
    assert proc.stdout is not None
    for line in iter(proc.stdout.readline, ""):
        if not line:
            break
        log_lines.append(line.rstrip("\n"))
        print(line, end="", flush=True)

    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Run '{run_name}' failed with exit code {ret}")

    metrics = parse_losses_from_output(log_lines)
    summary.append({
        "run": run_name,
        "encoder_use_condition": run_cfg["encoder_use_condition"],
        "decoder_z_memory_tokens": run_cfg["decoder_z_memory_tokens"],
        **metrics,
        "checkpoint": str(run_cfg["checkpoint_path"]),
        "loss_plot": str(run_cfg["loss_plot_path"]),
    })

print("\n=== Sweep summary ===")
try:
    import pandas as pd
    display(pd.DataFrame(summary).sort_values(by=["best_val", "final_val_total"], na_position="last"))
except Exception:
    for row in summary:
        print(row)

In [ ]:
import os
import subprocess

# Run training (prints epoch logs in notebook output)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

assert proc.stdout is not None
for line in iter(proc.stdout.readline, ""):
    if not line:
        break
    print(line, end="", flush=True)

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Training failed with exit code {ret}")

print("\nTraining complete.")

In [ ]:
artifacts = [
    train_cfg["checkpoint_path"],
    train_cfg["loss_plot_path"],
    train_cfg["cache_path"],
]

for p in artifacts:
    p = Path(p)
    print(f"{p}: {'exists' if p.exists() else 'missing'}")

/content/drive/MyDrive/Kilter_data/route_cvae.pt: exists
/content/drive/MyDrive/Kilter_data/route_cvae_loss_curve.png: exists
/content/drive/MyDrive/Kilter_data/preprocessed_routes_cache.pt: exists


In [ ]:
from pathlib import Path
from IPython.display import Image, display

loss_curve_path = Path(train_cfg.get("loss_plot_path", DATA_ROOT / "route_cvae_loss_curve.png"))

print(f"Loss curve path: {loss_curve_path}")
if not loss_curve_path.exists():
    raise FileNotFoundError(f"Loss curve image not found: {loss_curve_path}")

display(Image(filename=str(loss_curve_path)))

In [ ]:
# Collapse diagnostics: compare checkpoint reconstruction against z=0 on held-out validation batches.

from pathlib import Path
import torch
import torch.nn.functional as F

from data_preprocessing.route_preprocessing import split_route_samples
from model_training import (
    build_model_from_checkpoint,
    masked_mse,
    select_device,
    DecoderEOSIds,
    prepare_cvae_training_batch,
)
from model_training.route_vae_bottleneck import kl_divergence_loss

device = select_device()
print(f"Diagnostic device: {device}")


def _compute_recon_loss(out, prepared, *, ignore_index=-100):
    cat_targets = prepared["categorical_targets"]
    categorical_loss = sum(
        F.cross_entropy(
            out[f"{feat}_logits"].reshape(-1, out[f"{feat}_logits"].shape[-1]),
            cat_targets[f"{feat}_target"].reshape(-1),
            ignore_index=ignore_index,
        )
        for feat in ("type", "function", "role", "hole")
    )
    mask = prepared["valid_numeric_mask"]
    numeric_targets = prepared["numeric_targets"]
    numeric_loss = sum(
        masked_mse(out[f"{feat}_pred"], numeric_targets[f"{feat}_target"], mask)
        for feat in ("x", "y", "depth", "orientation_sin", "orientation_cos", "size")
    )
    recon = categorical_loss + train_cfg["numeric_weight"] * numeric_loss
    return recon, {
        "categorical": float(categorical_loss.detach().cpu()),
        "numeric": float(numeric_loss.detach().cpu()),
        "recon": float(recon.detach().cpu()),
    }


def _pick_checkpoint_path():
    candidates = [
        Path(train_cfg["checkpoint_path"]),
        Path(train_cfg["checkpoint_path"]).parent / "sweep_runs" / "route_cvae_disentangled.pt",
        Path(train_cfg["checkpoint_path"]).parent / "sweep_runs" / "route_cvae_disentangled_z8.pt",
        Path(train_cfg["checkpoint_path"]).parent / "sweep_runs" / "route_cvae_baseline.pt",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No checkpoint found for diagnostics.")


checkpoint_path = _pick_checkpoint_path()
print(f"Using checkpoint: {checkpoint_path}")

cache_payload = torch.load(train_cfg["cache_path"], map_location="cpu", weights_only=False)
samples = cache_payload["samples"]
vocabs = cache_payload["vocabs"]
train_samples, val_samples, test_samples = split_route_samples(
    samples, train_ratio=0.8, val_ratio=0.1, seed=train_cfg["seed"]
)

model, eos_ids = build_model_from_checkpoint(checkpoint_path, vocabs, device)
model.eval()

diag_batches = [val_samples[i:i + 16] for i in (0, 16) if val_samples[i:i + 16]]
if not diag_batches:
    raise ValueError("No validation samples available for diagnostics.")

for batch_index, batch_samples in enumerate(diag_batches, start=1):
    prepared = prepare_cvae_training_batch(batch_samples, eos_ids=eos_ids, device=device)

    with torch.no_grad():
        enc_out = model.encoder(
            prepared["encoder_batch"],
            angle=prepared["angle"],
            grade=prepared["grade"],
            grade_missing=prepared["grade_missing"],
        )
        bottleneck_out = model.bottleneck(enc_out["route_embedding"], sample_latent=False)

        base_out = model.decoder(
            prepared["decoder_input_batch"],
            z=bottleneck_out["z"],
            angle=prepared["angle"],
            grade=prepared["grade"],
            grade_missing=prepared["grade_missing"],
        )
        z0_out = model.decoder(
            prepared["decoder_input_batch"],
            z=torch.zeros_like(bottleneck_out["z"]),
            angle=prepared["angle"],
            grade=prepared["grade"],
            grade_missing=prepared["grade_missing"],
        )

        base_recon, base_stats = _compute_recon_loss(base_out, prepared)
        z0_recon, z0_stats = _compute_recon_loss(z0_out, prepared)
        kl = kl_divergence_loss(bottleneck_out["mu"], bottleneck_out["logvar"], reduction="mean")
        kl_per_dim = (-0.5 * (
            1 + bottleneck_out["logvar"]
            - bottleneck_out["mu"].pow(2)
            - bottleneck_out["logvar"].exp()
        )).mean(dim=0)

    print(f"Batch {batch_index} | ckpt={checkpoint_path.name}")
    print(
        f"  base total={base_recon + train_cfg['kl_beta'] * kl:.4f} recon={base_stats['recon']:.4f} kl={kl:.4f} "
        f"mu_norm={bottleneck_out['mu'].norm(dim=-1).mean().item():.4f} "
        f"mu_abs={bottleneck_out['mu'].abs().mean().item():.4f} "
        f"logvar_mean={bottleneck_out['logvar'].mean().item():.4f}"
    )
    print(
        f"  z0   total={z0_recon + train_cfg['kl_beta'] * kl:.4f} recon={z0_stats['recon']:.4f} "
        f"delta={(z0_recon - base_recon).item():+.4f} "
        f"cat_delta={(z0_stats['categorical'] - base_stats['categorical']):+.4f} "
        f"num_delta={(z0_stats['numeric'] - base_stats['numeric']):+.4f}"
    )
    top_k = torch.topk(kl_per_dim, k=min(4, kl_per_dim.numel())).values.tolist()
    print(
        f"  kl_per_dim: mean={kl_per_dim.mean().item():.6f} "
        f"min={kl_per_dim.min().item():.6f} max={kl_per_dim.max().item():.6f} top4={top_k}"
    )